In [4]:
#Importing modules and setting directory
import yaml
import os
import shutil
import subprocess

os.chdir('/glade/u/home/considine/')

In [5]:
#Class that creates wam2 experiments
class wam2experiment:

    #Constructor method, all inputs except coordinates are strings
    def __init__(self, cls, template, coordinates, exp_name):

        #Loading in configuration template
        with open(template, 'r') as f:
            cls.template = yaml.load(f, Loader=yaml.FullLoader)
        
        #Creating instances of variables unique to this experiment
        self.exp_abr = exp_name
        self.coordinates = coordinates  
        self.exp_name = f'/glade/derecho/scratch/considine/{exp_name}'
        self.config_folder = f'/glade/u/home/considine/{exp_name}_config'
        self.coord_name = None
            
        #Creating configuration folder
        x = True
        count = 0

        while x == True:
        
            try: 
                os.mkdir(self.config_folder)
            except Exception as e:
                print(f"An error occurred: {e}")
                count = count + 1
                self.config_folder = self.config_folder + str(count)
                continue

            x = False

        x = True
        count = 0

        #Creating folder for experiment output
        while x == True:
        
            try: 
                os.mkdir(self.exp_name)
            except Exception as e:
                print(f"An error occurred: {e}")
                count = count + 1
                self.config_folder = self.exp_name + str(count)
                continue

            x = False
        
        #Modifying the template for this experiment
        
        cls.template['output_folder'] = self.exp_name
        
        #Creating a configuration file for each set of coordinates
        for c in coordinates: 

            cls.template['tagging_region'] = c
            
            if c[0] < 0:
                self.coord_name = f'_n{str(abs(c[0])).zfill(3)}_'
            else:
                self.coord_name = f'_{str(c[0]).zfill(4)}_'
            if c[1] < 0:
                self.coord_name = f'{self.coord_name}n{str(abs(c[1])).zfill(3)}'
            else:
                self.coord_name = f'{self.coord_name}{str(c[1]).zfill(4)}'
                
            cls.template['output_folder'] = f'{self.exp_name}/output{self.coord_name}'
            
            #Writing configuration file for this specific set of coordinates
            file_name = f'config{self.coord_name}.yaml'
        
            with open(file_name, "a") as f:
                file_contents = yaml.dump(cls.template)
                f.write(file_contents)

            shutil.move(file_name, self.config_folder) #Moving file into configuration folder

    #Method to run experiment
    def run_experiment(self):

        #Creating folder for job and error files
        #x = True
        #count = 0
        
        #jobs_folder = self.exp_name + '_jobs'
        
        #while x == True:
            
            #try: 
                #os.mkdir(jobs_folder)
            #except Exception as e:
                #print(f"An error occurred: {e}")
                #count = count + 1
                #jobs_folder = jobs_folder + str(count)
                #continue

            #x = False

        os.chdir(self.config_folder)
        
        derecho_job = ['#!/bin/bash', #0
                 '#PBS -A WYOM0161', #1
                 '#PBS -l walltime=07:00:00', #2
                 '#PBS -q main', #3
                 '#PBS -l select=1:ncpus=1:mem=4GB', #4
                 '#PBS -N ', #5
                 '#PBS -e ', #6
                 '#PBS -o ', #7
                 'module load conda', #8
                 'conda activate wamenv', #9
                 'wam2layers track '] #10
        
        for file in os.listdir(self.config_folder):
            
            if file.startswith('c') == False:
                continue
        
            job_name = 'job'+ file[6:16]
           
            derecho_job[5] = f'#PBS -N {job_name}'
            derecho_job[6] = f'#PBS -e {job_name}_e.txt'
            derecho_job[7] = f'#PBS -o {job_name}_o.txt'
            derecho_job[10] = f'wam2layers track {self.config_folder}/{file}'

            with open(f'{job_name}.sh',"w") as f:
                f.write('\n'.join(derecho_job))

            subprocess.call(f'qsub {job_name}.sh',shell=True)
            
        return None
        

In [28]:
def rerun_experiment(exp_path,exp_name):
        empty_folders = []
        
        config_folder = f'/glade/u/home/considine/{exp_name}-config'

        for item in os.listdir(exp_path):
            
            if item.startswith('o') == False:
                continue

            listed_dir = os.listdir(exp_path+'/'+item)
            
            if len(listed_dir) == 0:
                empty_folders.append(item)

        os.chdir(config_folder)
    
        i = 0
        for folder in empty_folders:
            folder = folder.replace('output','config')
            folder = folder + '.yaml'
            empty_folders[i] = folder
            i = i + 1

        derecho_job = ['#!/bin/bash', #0
                 '#PBS -A WYOM0161', #1
                 '#PBS -l walltime=07:00:00', #2
                 '#PBS -q main', #3
                 '#PBS -l select=1:ncpus=1:mem=4GB', #4
                 '#PBS -N ', #5
                 '#PBS -e ', #6
                 '#PBS -o ', #7
                 'module load conda', #8
                 'conda activate wamenv', #9
                 'wam2layers track '] #10
            
        for file in empty_folders:
            
            job_name = 'job' + file[6:16]
           
            derecho_job[5] = f'#PBS -N {job_name}'
            derecho_job[6] = f'#PBS -e {job_name}_e.txt'
            derecho_job[7] = f'#PBS -o {job_name}_o.txt'
            derecho_job[10] = f'wam2layers track {config_folder}/{file}'
            
            with open(f'{job_name}.sh',"w") as f:
                f.write('\n'.join(derecho_job))

            subprocess.call(f'qsub {job_name}.sh',shell=True)
            
        return None

In [2]:
def create_coordinates(x):
    
    coordinates = []
    longitude = list(range(-180,179,x))
    latitude = list(range(-80,79,x))

    for w in longitude:

        for s in latitude: 
            e = w+x
            n = s+x
            
            coordinates.append([w,s,e,n])
            
    return coordinates

In [ ]:
#Running experiment
coordinates = [[128, 56, 132, 60]]

test = wam2experiment(wam2experiment,'2011-2012.yaml', coordinates, 'screwup')
test.run_experiment()

#Finding missing data
folder_names = []

for c in coordinates: 
    
    if c[0] < 0:
        coord_name = '_n'+str(abs(c[0])).zfill(3)+'_'
    else:
        coord_name = '_'+str(c[0]).zfill(4)+'_'
    if c[1] < 0:
        coord_name = coord_name + 'n'+str(abs(c[1])).zfill(3)
    else:
        coord_name = coord_name + str(c[1]).zfill(4)
                
    folder_names.append('output'+coord_name)
    
missing_data = []

for folder in folder_names:
    if folder in os.listdir('/glade/derecho/scratch/considine/2011-2012-forward'):
        continue
    else:
        missing_data.append(folder)
print(missing_data)

missing_coordinates = []

for folder in missing_data:
    
    folder = folder.replace('output_','')
    folder = folder.replace('n','-')
    folder = folder.rsplit('_')
    folder[0] = int(folder[0])
    folder[1] = int(folder[1])
    folder.append(folder[0] + 4)
    folder.append(folder[1] + 4)
    missing_coordinates.append(folder)


#Running simulations for missing data
missing = wam2experiment(wam2experiment,'2011-2012.yaml', missing_coordinates, '11_12_f_missing')
missing.run_experiment()

Notes

We will run wam2layers in this directory: /glade/work/considine/conda-envs/wamenv

module load conda
conda activate wamenv
wam2layers track <<path to config file>>

#!\bin\bash

#Need to submit on derecho computer
1990-2007 2007-2024



empty_folders = []

for folder in os.listdir('/glade/derecho/scratch/considine/2011-2012-forward'):
    if folder.startswith('o') == False:
        continue
    listed_dir = os.listdir('/glade/derecho/scratch/considine/2011-2012-forward/'+folder)
    if len(listed_dir) == 0:
        empty_folders.append(folder)

e_folders = empty_folders
empty_coordinates = []

for folder in e_folders:
    
    folder = folder.replace('output_','')
    folder = folder.replace('n','-')
    folder = folder.rsplit('_')
    folder[0] = int(folder[0])
    folder[1] = int(folder[1])
    folder.append(folder[0] + 4)
    folder.append(folder[1] + 4)
    empty_coordinates.append(folder)

empty = wam2experiment(wam2experiment,'2011-2012.yaml', empty_coordinates, '11_12_f_empty')
empty.run_experiment()